# Temporal modeling experiment — V2

Secondary longitudinal specification after V1. Compare the persistence baseline, a financial V2 Ridge logistic regression, and a financial + sporting V2 Ridge logistic regression using the same expanding-window temporal validation.

V2 adds current loss status, previous loss rate, and one-year changes in the three financial indicators. The target, regularization, C grid, external folds, inner temporal selection, threshold, and 2020 sensitivity remain unchanged from V1.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score

df = pd.read_csv('../data/processed/modeling_data_v2.csv')

## Define V2 features and common comparison sample

In [2]:
financial_v2_features = [
    'current_liquidity',
    'liability_ratio',
    'return_on_assets',
    'loss_current_year',
    'previous_loss_rate',
    'delta_current_liquidity',
    'delta_liability_ratio',
    'delta_return_on_assets',
]

sporting_features = [
    'division',
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
    'promoted',
    'relegated',
]

common_sample = df.dropna(
    subset=financial_v2_features + sporting_features
).copy()

common_sample.shape

(262, 42)

In [3]:
common_sample.groupby('fiscal_year')['loss_next_year'].agg(
    observations='count',
    losses='sum',
    loss_rate='mean',
)

,observations,losses,loss_rate
fiscal_year,,,
2017,31,16,0.516129
2018,32,17,0.531250
2019,32,23,0.718750
2020,33,15,0.454545
2021,35,15,0.428571
2022,35,14,0.400000
2023,33,24,0.727273
2024,31,20,0.645161


## Model specification

Continuous variables are standardized inside each training fold. `loss_current_year`, `promoted`, and `relegated` remain binary. `division` is one-hot encoded with Série A as the reference category. The Ridge logistic regression and `C` grid are unchanged from V1.

In [4]:
financial_continuous = [
    'current_liquidity',
    'liability_ratio',
    'return_on_assets',
    'previous_loss_rate',
    'delta_current_liquidity',
    'delta_liability_ratio',
    'delta_return_on_assets',
]

financial_binary = ['loss_current_year']

full_continuous = financial_continuous + [
    'points_per_game',
    'goal_difference_per_game',
    'relative_league_position',
]

full_categorical = ['division']
full_binary = financial_binary + ['promoted', 'relegated']

C_GRID = [0.01, 0.1, 1, 10, 100]

In [5]:
def build_pipeline(model_type, C):
    if model_type == 'financial':
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', StandardScaler(), financial_continuous),
                ('binary', 'passthrough', financial_binary),
            ],
            remainder='drop',
        )
    else:
        preprocessor = ColumnTransformer(
            transformers=[
                ('continuous', StandardScaler(), full_continuous),
                (
                    'division',
                    OneHotEncoder(
                        categories=[[1, 2, 3, 4]],
                        drop='first',
                        handle_unknown='ignore',
                    ),
                    full_categorical,
                ),
                ('binary', 'passthrough', full_binary),
            ],
            remainder='drop',
        )

    return Pipeline([
        ('preprocess', preprocessor),
        (
            'model',
            LogisticRegression(
                C=C,
                solver='liblinear',
                max_iter=1000,
            ),
        ),
    ])

## Temporal selection of C

For each external fold, `C` is selected using only earlier feature years. Internal validation also follows an expanding window. ROC-AUC is calculated separately in each internal validation year, and `C` is selected by the mean of those temporal AUCs. In an exact tie, the smaller `C` is chosen.

In [6]:
def select_c(train_data, model_type):
    years = sorted(train_data['fiscal_year'].unique())
    tuning_rows = []

    for C in C_GRID:
        fold_aucs = []
        validation_years = []

        for validation_year in years:
            inner_train = train_data[
                train_data['fiscal_year'] < validation_year
            ]
            inner_validation = train_data[
                train_data['fiscal_year'] == validation_year
            ]

            if inner_train['fiscal_year'].nunique() < 2:
                continue

            if inner_train['loss_next_year'].nunique() < 2:
                continue

            if inner_validation['loss_next_year'].nunique() < 2:
                continue

            model = build_pipeline(model_type, C)
            model.fit(inner_train, inner_train['loss_next_year'])

            probabilities = model.predict_proba(inner_validation)[:, 1]
            fold_auc = roc_auc_score(
                inner_validation['loss_next_year'],
                probabilities,
            )

            fold_aucs.append(fold_auc)
            validation_years.append(validation_year)

        tuning_rows.append({
            'C': C,
            'mean_inner_roc_auc': np.mean(fold_aucs),
            'std_inner_roc_auc': np.std(fold_aucs, ddof=0),
            'validation_years': validation_years,
        })

    tuning = pd.DataFrame(tuning_rows)
    tuning = tuning.sort_values(
        ['mean_inner_roc_auc', 'C'],
        ascending=[False, True],
    ).reset_index(drop=True)

    return tuning.loc[0, 'C'], tuning

## Expanding-window external experiment

The external folds are unchanged from V1 and are defined by feature year. `fiscal_year = 2020` predicts `target_year = 2021`, through `fiscal_year = 2024` predicting `target_year = 2025`.

In [7]:
def run_experiment(data, first_test_year):
    prediction_rows = []
    tuning_rows = []

    test_years = [
        year
        for year in sorted(data['fiscal_year'].unique())
        if year >= first_test_year
    ]

    for test_year in test_years:
        train = data[data['fiscal_year'] < test_year].copy()
        test = data[data['fiscal_year'] == test_year].copy()

        baseline_predictions = test['loss_current_year'].astype(int)

        for row_index, prediction in zip(test.index, baseline_predictions):
            prediction_rows.append({
                'club': test.loc[row_index, 'club'],
                'fiscal_year': test_year,
                'target_year': test.loc[row_index, 'target_year'],
                'model': 'baseline',
                'y_true': int(test.loc[row_index, 'loss_next_year']),
                'score': float(prediction),
                'prediction': int(prediction),
            })

        for model_type, model_name in [
            ('financial', 'financial_v2_ridge'),
            ('full', 'financial_sporting_v2_ridge'),
        ]:
            best_C, tuning = select_c(train, model_type)

            for _, tuning_row in tuning.iterrows():
                tuning_rows.append({
                    'test_feature_year': test_year,
                    'model': model_name,
                    'C': tuning_row['C'],
                    'mean_inner_roc_auc': tuning_row['mean_inner_roc_auc'],
                    'std_inner_roc_auc': tuning_row['std_inner_roc_auc'],
                    'selected': tuning_row['C'] == best_C,
                    'validation_years': tuning_row['validation_years'],
                })

            model = build_pipeline(model_type, best_C)
            model.fit(train, train['loss_next_year'])

            probabilities = model.predict_proba(test)[:, 1]
            predictions = (probabilities >= 0.5).astype(int)

            for row_index, probability, prediction in zip(
                test.index,
                probabilities,
                predictions,
            ):
                prediction_rows.append({
                    'club': test.loc[row_index, 'club'],
                    'fiscal_year': test_year,
                    'target_year': test.loc[row_index, 'target_year'],
                    'model': model_name,
                    'y_true': int(test.loc[row_index, 'loss_next_year']),
                    'score': float(probability),
                    'prediction': int(prediction),
                })

    predictions = pd.DataFrame(prediction_rows)
    tuning_results = pd.DataFrame(tuning_rows)

    return predictions, tuning_results

## Main V2 experiment

In [8]:
predictions, tuning_results = run_experiment(
    common_sample,
    first_test_year=2020,
)

In [9]:
tuning_results[tuning_results['selected']].reset_index(drop=True)

,test_feature_year,model,C,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2020,financial_v2_ridge,1.0,0.893720,0.000000,True,[2019]
1,2020,financial_sporting_v2_ridge,100.0,0.801932,0.000000,True,[2019]
2,2021,financial_v2_ridge,0.1,0.731079,0.138486,True,"[2019, 2020]"
3,2021,financial_sporting_v2_ridge,100.0,0.671337,0.130596,True,"[2019, 2020]"
4,2022,financial_v2_ridge,1.0,0.697413,0.144106,True,"[2019, 2020, 2021]"
5,2022,financial_sporting_v2_ridge,100.0,0.687558,0.109071,True,"[2019, 2020, 2021]"
6,2023,financial_v2_ridge,1.0,0.695679,0.124836,True,"[2019, 2020, 2021, 2022]"
7,2023,financial_sporting_v2_ridge,100.0,0.663627,0.103152,True,"[2019, 2020, 2021, 2022]"
8,2024,financial_v2_ridge,1.0,0.696358,0.111665,True,"[2019, 2020, 2021, 2022, 2023]"
9,2024,financial_sporting_v2_ridge,1.0,0.662382,0.096043,True,"[2019, 2020, 2021, 2022, 2023]"


## Pooled out-of-time performance — complementary

These metrics concatenate predictions from different external folds. They are retained as complementary diagnostics, not as the principal ROC-AUC summary.

In [10]:
pooled_results = (
    predictions
    .groupby('model')
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'pooled_roc_auc': roc_auc_score(group['y_true'], group['score']),
            'pooled_balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'pooled_accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

pooled_results

,model,observations,pooled_roc_auc,pooled_balanced_accuracy,pooled_accuracy
0,baseline,167.0,0.632696,0.632696,0.634731
1,financial_sporting_v2_ridge,167.0,0.630178,0.649094,0.652695
2,financial_v2_ridge,167.0,0.625000,0.600403,0.604790


## Performance by external feature year

In [11]:
yearly_results = (
    predictions
    .groupby(['fiscal_year', 'model'])
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'roc_auc': roc_auc_score(group['y_true'], group['score']),
            'balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

yearly_results

,fiscal_year,model,observations,roc_auc,balanced_accuracy,accuracy
0,2020,baseline,33.0,0.566667,0.566667,0.545455
1,2020,financial_sporting_v2_ridge,33.0,0.540741,0.533333,0.515152
2,2020,financial_v2_ridge,33.0,0.551852,0.538889,0.515152
3,2021,baseline,35.0,0.600000,0.600000,0.600000
4,2021,financial_sporting_v2_ridge,35.0,0.720000,0.716667,0.714286
5,2021,financial_v2_ridge,35.0,0.626667,0.533333,0.514286
6,2022,baseline,35.0,0.738095,0.738095,0.742857
7,2022,financial_sporting_v2_ridge,35.0,0.591837,0.619048,0.628571
8,2022,financial_v2_ridge,35.0,0.690476,0.702381,0.714286
9,2023,baseline,33.0,0.618056,0.618056,0.545455


## Principal temporal ROC-AUC summary

The principal ROC-AUC result is the simple mean of the external yearly AUCs. Each test year receives equal weight.

In [12]:
temporal_auc_results = (
    yearly_results
    .groupby('model')['roc_auc']
    .agg(
        mean_roc_auc='mean',
        std_roc_auc='std',
        min_roc_auc='min',
        max_roc_auc='max',
        folds='count',
    )
    .reset_index()
)

temporal_auc_results

,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,baseline,0.644109,0.071292,0.566667,0.738095,5
1,financial_sporting_v2_ridge,0.654691,0.093776,0.540741,0.772727,5
2,financial_v2_ridge,0.660886,0.072612,0.551852,0.736364,5


## Sensitivity analysis without feature year 2020

Only rows whose feature year is 2020 are removed. Earlier rows whose target happens to be 2020 remain valid.

In [13]:
sample_without_2020 = common_sample[
    common_sample['fiscal_year'] != 2020
].copy()

predictions_without_2020, tuning_without_2020 = run_experiment(
    sample_without_2020,
    first_test_year=2021,
)

In [14]:
yearly_results_without_2020 = (
    predictions_without_2020
    .groupby(['fiscal_year', 'model'])
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'roc_auc': roc_auc_score(group['y_true'], group['score']),
            'balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

yearly_results_without_2020

,fiscal_year,model,observations,roc_auc,balanced_accuracy,accuracy
0,2021,baseline,35.0,0.600000,0.600000,0.600000
1,2021,financial_sporting_v2_ridge,35.0,0.653333,0.591667,0.571429
2,2021,financial_v2_ridge,35.0,0.583333,0.558333,0.542857
3,2022,baseline,35.0,0.738095,0.738095,0.742857
4,2022,financial_sporting_v2_ridge,35.0,0.636054,0.583333,0.571429
5,2022,financial_v2_ridge,35.0,0.731293,0.714286,0.714286
6,2023,baseline,33.0,0.618056,0.618056,0.545455
7,2023,financial_sporting_v2_ridge,33.0,0.680556,0.680556,0.636364
8,2023,financial_v2_ridge,33.0,0.662037,0.638889,0.575758
9,2024,baseline,31.0,0.697727,0.697727,0.741935


In [15]:
temporal_auc_results_without_2020 = (
    yearly_results_without_2020
    .groupby('model')['roc_auc']
    .agg(
        mean_roc_auc='mean',
        std_roc_auc='std',
        min_roc_auc='min',
        max_roc_auc='max',
        folds='count',
    )
    .reset_index()
)

temporal_auc_results_without_2020

,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,baseline,0.663470,0.065405,0.600000,0.738095,4
1,financial_sporting_v2_ridge,0.669759,0.031986,0.636054,0.709091,4
2,financial_v2_ridge,0.679393,0.073044,0.583333,0.740909,4


In [16]:
pooled_results_without_2020 = (
    predictions_without_2020
    .groupby('model')
    .apply(
        lambda group: pd.Series({
            'observations': len(group),
            'pooled_roc_auc': roc_auc_score(group['y_true'], group['score']),
            'pooled_balanced_accuracy': balanced_accuracy_score(
                group['y_true'], group['prediction']
            ),
            'pooled_accuracy': accuracy_score(
                group['y_true'], group['prediction']
            ),
        }),
        include_groups=False,
    )
    .reset_index()
)

pooled_results_without_2020

,model,observations,pooled_roc_auc,pooled_balanced_accuracy,pooled_accuracy
0,baseline,134.0,0.657983,0.657983,0.656716
1,financial_sporting_v2_ridge,134.0,0.658657,0.633506,0.641791
2,financial_v2_ridge,134.0,0.662924,0.638895,0.641791


In [17]:
tuning_without_2020[
    tuning_without_2020['selected']
].reset_index(drop=True)

,test_feature_year,model,C,mean_inner_roc_auc,std_inner_roc_auc,selected,validation_years
0,2021,financial_v2_ridge,1.0,0.893720,0.000000,True,[2019]
1,2021,financial_sporting_v2_ridge,100.0,0.801932,0.000000,True,[2019]
2,2022,financial_v2_ridge,1.0,0.738527,0.155193,True,"[2019, 2021]"
3,2022,financial_sporting_v2_ridge,100.0,0.727633,0.074300,True,"[2019, 2021]"
4,2023,financial_v2_ridge,1.0,0.736115,0.126761,True,"[2019, 2021, 2022]"
5,2023,financial_sporting_v2_ridge,100.0,0.697107,0.074458,True,"[2019, 2021, 2022]"
6,2024,financial_v2_ridge,1.0,0.717596,0.114368,True,"[2019, 2021, 2022, 2023]"
7,2024,financial_sporting_v2_ridge,1.0,0.697294,0.071263,True,"[2019, 2021, 2022, 2023]"


## Compare principal main and sensitivity results

In [18]:
comparison = pd.concat([
    temporal_auc_results.assign(sample='main'),
    temporal_auc_results_without_2020.assign(sample='without_2020'),
], ignore_index=True)

comparison[
    ['sample', 'model', 'mean_roc_auc', 'std_roc_auc', 'min_roc_auc', 'max_roc_auc', 'folds']
]

,sample,model,mean_roc_auc,std_roc_auc,min_roc_auc,max_roc_auc,folds
0,main,baseline,0.644109,0.071292,0.566667,0.738095,5
1,main,financial_sporting_v2_ridge,0.654691,0.093776,0.540741,0.772727,5
2,main,financial_v2_ridge,0.660886,0.072612,0.551852,0.736364,5
3,without_2020,baseline,0.663470,0.065405,0.600000,0.738095,4
4,without_2020,financial_sporting_v2_ridge,0.669759,0.031986,0.636054,0.709091,4
5,without_2020,financial_v2_ridge,0.679393,0.073044,0.583333,0.740909,4
